####Package Installation

In [ ]:
!pip install faiss-cpu langchain langchain-community langchain-google-genai pandas

####Imports

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"]=userdata.get('GOOGLE_API_KEY')

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

####Initiate the Vector Store

In [ ]:
pip install -qU langchain-huggingface

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
index = faiss.IndexFlatL2(len(embeddings.embed_query(" ")))
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

####Process CSV Data

In [ ]:
loader = CSVLoader(file_path='cleaned_googleplaystore.csv')
docs = loader.load_and_split()
##line docs = loader.load_and_split() reads your CSV file and creates a list called docs, where each element in the list represents a row from your CSV file, formatted as a document object that can be used in subsequent steps for tasks like creating embeddings and building a vector store.

####Add the data to the vector store

In [ ]:
vector_store.add_documents(documents=docs)

['7ab3dc42-2b2f-4174-a401-43bd89b9bf82']

####Create teh Retrieval Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

retriever = vector_store.as_retriever()

# Set up system prompt
system_prompt = (
    """
    You are an expert Applied AI Engineer specializing in app marketing and strategy.
Your task is to analyze app store data and provide actionable insights and recommendations.
Focus on identifying trends, popular categories, monetization strategies, user engagement factors, and potential market opportunities based on the provided data.
Provide clear, concise, and data-driven insights.
    """
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),

])

# Create the question-answer chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

####Query the RAG bot with a question based on the CSV data

In [ ]:
answer= rag_chain.invoke({"input": "Give insights on the dataset"})
answer['answer']

'Here\'s an analysis of the provided app store data, focusing on trends, monetization, user engagement, and market opportunities:\n\n### Key Insights from the Dataset:\n\n1.  **Strong Demand for Free Educational/Business Tools:**\n    *   Apps like "DataCamp - Learn R, Python & SQL" and "Microsoft Power BI" demonstrate significant user acquisition (100,000+ installs) when offered for free. These apps cater to professional development and skill-building (coding, data analytics, business intelligence).\n    *   Their high ratings (4.4-4.5) and substantial review counts (944-1819) indicate strong user satisfaction and engagement.\n\n2.  **Monetization Strategy - Free-to-Use with Potential for Upsell/Subscription:**\n    *   The most successful apps in terms of installs are "Free" with a price of $0.0. This suggests that a free entry point is crucial for widespread adoption in these categories.\n    *   For apps like DataCamp, the business model likely involves in-app subscriptions or prem

#### Save the response to a JSON file for insights

In [ ]:
import json

response_data = {
    "input": answer['input'],
    "context": [doc.page_content for doc in answer['context']], # Extracting content from Document objects
    "answer": answer['answer']
}

with open('app_insights.json', 'w') as f:
    json.dump(response_data, f, indent=4)

print("Response saved to app_insights.json")

Response saved to app_insights.json


In [ ]:
answer= rag_chain.invoke({"input": "Give me some app recommendations "})
answer['answer']

"Based on the app store data provided, and assuming you're looking for productivity apps, here are some recommendations with their key strengths and considerations:\n\n1.  **Pocket RxTx Free**\n    *   **Why it's recommended:** This app has a massive user base (100,000+ installs) and a solid rating of 4.1 stars from over 4,500 reviews. This indicates a widely adopted and generally well-received app. It's a good choice if you're looking for an established tool that many people find useful.\n    *   **Considerations:** The last update was in March 2018, which is quite old. This might mean it lacks modern features, may not be fully optimized for the latest Android versions, or could have security implications if it handles sensitive data.\n\n2.  **Krypton by krypt.co**\n    *   **Why it's recommended:** With an excellent rating of 4.6 stars, this app is highly regarded by its users. It also has a more recent update (July 2018) compared to Pocket RxTx Free, suggesting it's slightly more ac

In [ ]:
import json

response_data = {
    "input": answer['input'],
    "context": [doc.page_content for doc in answer['context']], # Extracting content from Document objects
    "answer": answer['answer']
}

with open('recommendations.json', 'w') as f:
    json.dump(response_data, f, indent=4)

print("Response saved to recommendations.json")

Response saved to recommendations.json


1. The Current Implementation Assumes a Clean, Parseable Response:

Imagine you ask the LLM for app recommendations based on your CSV data, and you expect the output to be a structured list of app names and their ratings, like this:


```
# This is formatted as code
[
  {"name": "Pocket RxTx Free", "rating": 4.1},
  {"name": "Krypton by krypt.co", "rating": 4.6},
  {"name": "iReadMe", "rating": 5.0}
]
```


Your current code is written to easily read this format and extract the app names and ratings. It assumes the LLM will always return something exactly like this.

2. The Problem: LLMs Don't Always Return Parseable Responses:

Sometimes, the LLM might return something different, like:
```
Missing information: "Here are some apps: Pocket RxTx Free, Krypton, iReadMe." (No ratings)

Incorrect format: "Pocket RxTx Free (Rating: 4.1), Krypton by krypt.co (4.6 stars), iReadMe has a 5.0 rating." (Inconsistent formatting)

Extra text: "Based on your request, here are some app recommendations. [The list of apps in the correct format] I hope this helps!" (Extra conversational text)

Hallucinated data: Including apps that weren't in the original data.

In any of these cases, your current code, which expects a specific structure, might fail to extract the information or might even crash.
```

3. Adding a Validation Layer (using Pydantic):

This is where Pydantic comes in. You would define a Pydantic model that represents the structure you expect the LLM's output to have:
```
from pydantic import BaseModel
from typing import List

class AppRecommendation(BaseModel):
    name: str
    rating: float

class AppRecommendationsList(BaseModel):
    recommendations: List[AppRecommendation]

# In your code, after getting the LLM response:
try:
    # Assuming the LLM returns a dictionary like {"recommendations": [...]}
    validated_response = AppRecommendationsList.model_validate(llm_response)
    # Now you can safely access validated_response.recommendations
except ValidationError as e:
    print(f"Validation failed: {e}")
```
This is where the retry mechanism would be triggered

In this example, AppRecommendation defines the structure for a single app, and AppRecommendationsList defines that the overall response should contain a list of these. If the LLM's response doesn't match this structure (e.g., missing the "rating" field, or "rating" is not a number), Pydantic will raise a ValidationError.

4. Adding a Retry Mechanism:

If the Pydantic validation fails, the retry mechanism would kick in. This could involve:

Logging the error: Record that the LLM returned an unparseable response.
Retrying the query: Send the same query to the LLM again.

Modifying the prompt (optional but helpful): If retrying the same query doesn't work, you could try modifying the prompt to be more explicit about the desired output format. For example, you could add to your prompt: "Please provide the app recommendations as a JSON list with each item having 'name' and 'rating' fields."

Setting a limit: Don't retry indefinitely. After a certain number of failed attempts, you might log a final error and handle the situation gracefully (e.g., inform the user that the recommendations could not be generated).

By combining Pydantic validation and a retry mechanism, you make your system much more resilient to variations and errors in the LLM's output, ensuring that your insight generation process is more reliable.



Caching LLM Responses

Explanation:

Caching involves storing the results of previous LLM requests so that if the exact same request is made again, you can return the stored response instead of making a new API call. This is particularly useful for queries that are likely to be repeated frequently.

Example:
Imagine you have a common query like "What are the top 3 productivity apps in the business category?". If many users or parts of your application ask this same question, you can:

1. The first time this query is made, send it to the LLM API.
2. Receive the response from the LLM (e.g., "Pocket RxTx Free, Krypton by krypt.co, iReadMe").
3. Store this query and its response in a cache (e.g., a dictionary, a database, or a dedicated caching system like Redis).
4. The next time the exact same query ("What are the top 3 productivity apps in the business category?") is received, check your cache first.
5. If the query is found in the cache, return the stored response immediately without calling the LLM API.
6. If the query is not in the cache, send it to the LLM API, get the response, and store it in the cache for future use.

Benefits of Caching:

1. Reduced API Costs: You pay for fewer API calls.
2. Faster Response Times: Retrieving from a cache is much faster than waiting for an LLM API to process a request.
3. Reduced Load on LLM API: Less traffic to the LLM provider's servers.


Considerations for Caching:

1. Cache Invalidation: How do you handle situations where the underlying data changes? If new apps are added or ratings change, your cached response for "top 3 productivity apps" might become outdated. You need a strategy to update or clear cached entries when necessary.
2. Cache Size: Caches have limited capacity. You need a policy for deciding which items to remove when the cache is full (e.g., Least Recently Used - LRU).
3. Query Variation: Caching works best for exact matches. Slight variations in wording (e.g., "best productivity apps for business" instead of "top 3 productivity apps in the business category") would result in a cache miss.

Rate Limiting API Requests

Explanation:

Rate limiting is a technique to control the number of requests your application sends to an API within a specific time window. LLM providers often have rate limits to prevent abuse and ensure fair usage of their services. Exceeding the rate limit can result in errors or even temporary bans. Implementing your own rate limiting helps you stay within the provider's limits and avoid these issues, as well as manage your spending.

Example:

Suppose your LLM provider allows you to make a maximum of 100 requests per minute. You can implement a rate limiting mechanism in your application:

1. Maintain a counter of requests made within the current minute.
2. Before sending a request to the LLM API, check the counter.
3. If the counter is below 100, send the request and increment the counter.
4. If the counter has reached 100, pause sending further requests until the next minute begins. You might queue the requests to be sent later or inform the user to try again.

Benefits of Rate Limiting:

1. Avoids Exceeding API Limits: Prevents errors and potential service disruptions.
2. Cost Control: Helps you stay within a budget by limiting the number of requests you make, especially if you have a per-request pricing model.
3. Improved System Stability: Prevents your application from overwhelming the LLM API or your own infrastructure with too many requests.

Considerations for Rate Limiting:

1. Provider's Limits: You need to know the specific rate limits imposed by your LLM provider.
2. Implementation Strategy: You can use various algorithms for rate limiting (e.g., Leaky Bucket, Token Bucket). The best choice depends on your specific needs.
3. Handling Exceeded Limits: Decide how your application should behave when the rate limit is reached (e.g., retry after a delay, queue requests, return an error).